# Translation Pipeline Review

This notebook is for auditing the multilingual translation pipeline outputs.

Default run directory:

- `/scratch/ziv_baretto/Research_X/Partisan-Discourse-on-X-English-/Additional_Steps/Translating/Data/full_multilingual_en_20260403_detached`

What this notebook helps with:

- inspect branch sizes and file paths
- sample English, native-script, and Roman/Hinglish routed rows
- inspect translated outputs
- inspect low-cosine examples
- look up a specific `row_id` across stages
- search tweets by text
- inspect the final merged English output when it exists

In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
from IPython.display import display, Markdown, HTML

BASE_DIR = Path("/scratch/ziv_baretto/Research_X/Partisan-Discourse-on-X-English-/Additional_Steps/Translating")
RUN_DIR = Path("/scratch/ziv_baretto/Research_X/Partisan-Discourse-on-X-English-/Additional_Steps/Translating/Data/full_multilingual_en_20260403_detached")
ROUTED_DIR = RUN_DIR / "routed"
TRANSLATED_DIR = RUN_DIR / "translated"
LOG_DIR = RUN_DIR / "logs"

FILES = {
    "english_routed": ROUTED_DIR / "english_tweets.csv",
    "native_routed": ROUTED_DIR / "hindi_script_tweets.csv",
    "roman_routed": ROUTED_DIR / "hinglish_roman_tweets.csv",
    "native_translated": TRANSLATED_DIR / "hindi_script_en.csv",
    "roman_translated": TRANSLATED_DIR / "hinglish_roman_en.csv",
    "native_lowcos": TRANSLATED_DIR / "hindi_script_lowcos.csv",
    "roman_lowcos": TRANSLATED_DIR / "hinglish_roman_lowcos.csv",
    "merged": RUN_DIR / "retweet_data_combined_labeled_by_retweet_en.csv",
    "route_log": LOG_DIR / "01_route.log",
    "native_log": LOG_DIR / "02_translate_native.log",
    "roman_log": LOG_DIR / "03_translate_roman.log",
    "merge_log": LOG_DIR / "04_merge.log",
}

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 50)

display(
    HTML(
        '''
        <style>
        table.dataframe td, table.dataframe th {
            white-space: pre-wrap !important;
            max-width: none !important;
            overflow: visible !important;
            text-overflow: clip !important;
        }
        </style>
        '''
    )
)

summary_rows = []
for key, path in FILES.items():
    summary_rows.append(
        {
            "name": key,
            "path": str(path),
            "exists": path.exists(),
            "size_mb": round(path.stat().st_size / (1024 * 1024), 2) if path.exists() else None,
        }
    )
display(pd.DataFrame(summary_rows))

,name,path,exists,size_mb
0,english_routed,/scratch/ziv_baretto/Research_X/Partisan-Discourse-on-X-English-/Additional_Steps/Translating/Data/full_multilingual_en_20260403_detached/routed/english_tweets.csv,True,1464.40
1,native_routed,/scratch/ziv_baretto/Research_X/Partisan-Discourse-on-X-English-/Additional_Steps/Translating/Data/full_multilingual_en_20260403_detached/routed/hindi_script_tweets.csv,True,3652.96
2,roman_routed,/scratch/ziv_baretto/Research_X/Partisan-Discourse-on-X-English-/Additional_Steps/Translating/Data/full_multilingual_en_20260403_detached/routed/hinglish_roman_tweets.csv,True,12.44
3,native_translated,/scratch/ziv_baretto/Research_X/Partisan-Discourse-on-X-English-/Additional_Steps/Translating/Data/full_multilingual_en_20260403_detached/translated/hindi_script_en.csv,True,4462.78
4,roman_translated,/scratch/ziv_baretto/Research_X/Partisan-Discourse-on-X-English-/Additional_Steps/Translating/Data/full_multilingual_en_20260403_detached/translated/hinglish_roman_en.csv,True,16.14
5,native_lowcos,/scratch/ziv_baretto/Research_X/Partisan-Discourse-on-X-English-/Additional_Steps/Translating/Data/full_multilingual_en_20260403_detached/translated/hindi_script_lowcos.csv,True,421.64
6,roman_lowcos,/scratch/ziv_baretto/Research_X/Partisan-Discourse-on-X-English-/Additional_Steps/Translating/Data/full_multilingual_en_20260403_detached/translated/hinglish_roman_lowcos.csv,True,1.59
7,merged,/scratch/ziv_baretto/Research_X/Partisan-Discourse-on-X-English-/Additional_Steps/Translating/Data/full_multilingual_en_20260403_detached/retweet_data_combined_labeled_by_retweet_en.csv,True,990.96
8,route_log,/scratch/ziv_baretto/Research_X/Partisan-Discourse-on-X-English-/Additional_Steps/Translating/Data/full_multilingual_en_20260403_detached/logs/01_route.log,True,0.01
9,native_log,/scratch/ziv_baretto/Research_X/Partisan-Discourse-on-X-English-/Additional_Steps/Translating/Data/full_multilingual_en_20260403_detached/logs/02_translate_native.log,True,0.92


In [2]:
def route_totals_from_log(route_log: Path) -> dict:
    text = route_log.read_text(errors="ignore")
    pat = re.compile(r"\[chunk \d+\] rows=\d+ total=\d+ english=(\d+) native=(\d+) roman=(\d+)")
    english = native = roman = 0
    for e, n, r in pat.findall(text):
        english += int(e)
        native += int(n)
        roman += int(r)
    return {
        "english": english,
        "native": native,
        "roman": roman,
        "total": english + native + roman,
    }


def last_translated_total(log_path: Path) -> int:
    if not log_path.exists():
        return 0
    text = log_path.read_text(errors="ignore")
    matches = re.findall(r"chunk \d+ processed \d+ rows; total (\d+)", text)
    return int(matches[-1]) if matches else 0


route_totals = route_totals_from_log(FILES["route_log"])
native_done = last_translated_total(FILES["native_log"])
roman_done = last_translated_total(FILES["roman_log"])

display(
    pd.DataFrame(
        [
            {"stage": "route_total", "rows": route_totals["total"]},
            {"stage": "english_routed", "rows": route_totals["english"]},
            {"stage": "native_routed", "rows": route_totals["native"]},
            {"stage": "roman_routed", "rows": route_totals["roman"]},
            {"stage": "native_translated_so_far", "rows": native_done},
            {"stage": "roman_translated_so_far", "rows": roman_done},
        ]
    )
)

,stage,rows
0,route_total,7115963
1,english_routed,3006222
2,native_routed,4078560
3,roman_routed,31181
4,native_translated_so_far,2560000
5,roman_translated_so_far,31181


In [3]:
def sample_csv(path: Path, n: int = 10, usecols=None, chunksize: int = 100_000, random_state: int = 42) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    rng = np.random.default_rng(random_state)
    samples = []
    for chunk in pd.read_csv(path, chunksize=chunksize, usecols=usecols):
        take = min(max(n, 1), len(chunk))
        samples.append(chunk.sample(n=take, random_state=int(rng.integers(0, 1_000_000_000))))
    if not samples:
        return pd.DataFrame(columns=usecols)
    out = pd.concat(samples, ignore_index=True)
    if len(out) > n:
        out = out.sample(n=n, random_state=random_state)
    return out.reset_index(drop=True)


def head_csv(path: Path, n: int = 5, usecols=None) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, nrows=n, usecols=usecols)


def lookup_row(row_id: int, files=None) -> dict:
    files = files or {
        "english_routed": FILES["english_routed"],
        "native_routed": FILES["native_routed"],
        "roman_routed": FILES["roman_routed"],
        "native_translated": FILES["native_translated"],
        "roman_translated": FILES["roman_translated"],
        "merged": FILES["merged"],
    }
    found = {}
    for name, path in files.items():
        if not path.exists():
            continue
        matches = []
        for chunk in pd.read_csv(path, chunksize=100_000):
            hit = chunk[chunk["row_id"] == row_id]
            if not hit.empty:
                matches.append(hit)
        if matches:
            found[name] = pd.concat(matches, ignore_index=True)
    return found


def search_text(path: Path, pattern: str, text_col: str = "tweet_original", n: int = 20, case: bool = False) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    matches = []
    flags = 0 if case else re.IGNORECASE
    for chunk in pd.read_csv(path, chunksize=100_000):
        series = chunk[text_col].fillna("").astype(str)
        hit = chunk[series.str.contains(pattern, regex=True, flags=flags)]
        if not hit.empty:
            matches.append(hit)
        if sum(len(df) for df in matches) >= n:
            break
    if not matches:
        return pd.DataFrame()
    out = pd.concat(matches, ignore_index=True)
    return out.head(n)

## Routed Branch Samples

These cells let you inspect what the router produced before translation.

In [4]:
route_cols = [
    "row_id",
    "translation_route",
    "lid_lang",
    "lid_score",
    "tweet_original",
    "tweet",
    "nllb_src_lang",
    "xlit_lang",
]

display(Markdown("### English branch sample"))
display(sample_csv(FILES["english_routed"], n=8, usecols=route_cols))

display(Markdown("### Native-script branch sample"))
display(sample_csv(FILES["native_routed"], n=8, usecols=route_cols))

display(Markdown("### Roman/Hinglish branch sample"))
display(sample_csv(FILES["roman_routed"], n=8, usecols=route_cols))

### English branch sample

,tweet,row_id,tweet_original,translation_route,lid_lang,lid_score,xlit_lang,nllb_src_lang
0,#StopAdani protesters disrupted play at the #AUSvIND Cricket match\nhttps://t.co/ldd84kY90K,1041710,#StopAdani protesters disrupted play at the #AUSvIND Cricket match\nhttps://t.co/ldd84kY90K,english,eng_Latn,0.997570,NaN,NaN
1,India &amp; COVID 19: Did @AIADMKOfficial handle #COVID19 situation poorly leading to spread of the disease in TN? Watch what DMK leader @Dayanidhi_Maran says on it at a time when Tamil Nadu is gearing up for polls amidst #COVID19. @bhupendrachaube\nFull Show: https://t.co/F6ujDpL9WR https://t.co/UXARDh3lCu,158161,India &amp; COVID 19: Did @AIADMKOfficial handle #COVID19 situation poorly leading to spread of the disease in TN? Watch what DMK leader @Dayanidhi_Maran says on it at a time when Tamil Nadu is gearing up for polls amidst #COVID19. @bhupendrachaube\nFull Show: https://t.co/F6ujDpL9WR https://t.co/UXARDh3lCu,english,eng_Latn,0.999942,NaN,NaN
2,Union Home Minister Amit Shah reviews prevailing threat scenario in country and emerging security challenges: MHA,5128908,Union Home Minister Amit Shah reviews prevailing threat scenario in country and emerging security challenges: MHA,english,eng_Latn,0.999372,NaN,NaN
3,Well done 🇮🇳 https://t.co/NTMYT3Gu1O,2227685,Well done 🇮🇳 https://t.co/NTMYT3Gu1O,english,other,0.961991,NaN,NaN
4,NDTV #GroundReport | Ankita Cremated: Power &amp; Politics And The Fight For Justice?\n\nWATCH: https://t.co/iafDqrY5Zv https://t.co/YpMHM4KAnP,4299407,NDTV #GroundReport | Ankita Cremated: Power &amp; Politics And The Fight For Justice?\n\nWATCH: https://t.co/iafDqrY5Zv https://t.co/YpMHM4KAnP,english,eng_Latn,0.992574,NaN,NaN
5,@AbhimanyuX_ I don't think so!\nBroken at the first place and mending things for whatever reason cannot bring those moments back which already have been lost . Forgetting past is another lie to self.. nobody forgets it completely.,7105084,@AbhimanyuX_ I don't think so!\nBroken at the first place and mending things for whatever reason cannot bring those moments back which already have been lost . Forgetting past is another lie to self.. nobody forgets it completely.,english,eng_Latn,0.999996,NaN,NaN
6,RT @hemantrajora_: NEW : No Regular quarantine for healthcare workers after performing duty in #COVID19 areas,5510136,RT @hemantrajora_: NEW : No Regular quarantine for healthcare workers after performing duty in #COVID19 areas,english,eng_Latn,0.999800,NaN,NaN
7,"Schedule of Union Home Minister Shri @AmitShah's public programs in West Bengal today, 2 April 2021.\n\n1) Pubic Meeting in Sitalkuchi at 11AM\n\n2) Pubic Meeting in Kalchini at 12:30PM\n\n3) Road Show in Baruipur Paschim at3PM \n\n4) Road Show in Arambag at 4:30PM https://t.co/Nq1kL8G3EK",2991396,"Schedule of Union Home Minister Shri @AmitShah's public programs in West Bengal today, 2 April 2021.\n\n1) Pubic Meeting in Sitalkuchi at 11AM\n\n2) Pubic Meeting in Kalchini at 12:30PM\n\n3) Road Show in Baruipur Paschim at3PM \n\n4) Road Show in Arambag at 4:30PM https://t.co/Nq1kL8G3EK",english,eng_Latn,1.000022,NaN,NaN


### Native-script branch sample

,tweet,row_id,tweet_original,translation_route,lid_lang,lid_score,xlit_lang,nllb_src_lang
0,महाराष्ट्र- कर्नाटक सीमा प्रश्न पेटवणं ही भाजपची राजकीय गरज आहे.कर्नाटकात निवडणुका होणार आहेत. कन्नड रक्षण वेदिके या संघटनेला बोम्मई सरकारचा छुपा पाठींबा दिसतो आहे. आणि आमचे मुख्यमंत्री शिंदे लाचारीत मग्न आहेत! \n#BJP #eknathshindeCM,5120331,महाराष्ट्र- कर्नाटक सीमा प्रश्न पेटवणं ही भाजपची राजकीय गरज आहे.कर्नाटकात निवडणुका होणार आहेत. कन्नड रक्षण वेदिके या संघटनेला बोम्मई सरकारचा छुपा पाठींबा दिसतो आहे. आणि आमचे मुख्यमंत्री शिंदे लाचारीत मग्न आहेत! \n#BJP #eknathshindeCM,hindi_script,mar_Deva,1.000018,NaN,mar_Deva
1,कानून बहुत घटिया है इसलिए 1 लाख नकली रेमडेसिविर बनाकर पूरे देश में बेच दिया\n\nनक्कालों मिलावटखोरों जमाखोरों मुनाफाखोरों और कालाबाजारियों की 100% चल-अचल संपत्ति जब्त करने तथा उन्हें आजीवन कारावास देने के लिए कानून बनाना नितांत आवश्यक है\n\n@AmitShah @blsanthosh\n@narendramodi @JPNadda,2444116,कानून बहुत घटिया है इसलिए 1 लाख नकली रेमडेसिविर बनाकर पूरे देश में बेच दिया\n\nनक्कालों मिलावटखोरों जमाखोरों मुनाफाखोरों और कालाबाजारियों की 100% चल-अचल संपत्ति जब्त करने तथा उन्हें आजीवन कारावास देने के लिए कानून बनाना नितांत आवश्यक है\n\n@AmitShah @blsanthosh\n@narendramodi @JPNadda,hindi_script,hin_Deva,0.894039,NaN,hin_Deva
2,#FuelPriceHike के बारे में मोदी के मंत्री @HardeepSPuri “चलो छोड़ो यार” कह के दरवाज़ा बन्ध किया। आम आदमी कैसे छोड़ो इस बात को मंत्रीजी? कुछ तो तमीज़ सिखाओ उन्हें @PMOIndia \nhttps://t.co/jMhQknwzYF,5734683,#FuelPriceHike के बारे में मोदी के मंत्री @HardeepSPuri “चलो छोड़ो यार” कह के दरवाज़ा बन्ध किया। आम आदमी कैसे छोड़ो इस बात को मंत्रीजी? कुछ तो तमीज़ सिखाओ उन्हें @PMOIndia \nhttps://t.co/jMhQknwzYF,hindi_script,kas_Deva,0.711697,NaN,kas_Deva
3,"“राजनीतिक स्वार्थ, कट्टरपन व अलगाव की भावना, भारत के प्रति शत्रुता तथा जागतिक वर्चस्व की महत्वाकांक्षा, इनका एक अजीब सम्मिश्रण भारत की राष्ट्रीय एकात्मता के विरुद्ध काम कर रहा है।” -मोहनजी भागवत \n#RSSVijayaDashami https://t.co/Bn5cKzjeYQ",229866,"“राजनीतिक स्वार्थ, कट्टरपन व अलगाव की भावना, भारत के प्रति शत्रुता तथा जागतिक वर्चस्व की महत्वाकांक्षा, इनका एक अजीब सम्मिश्रण भारत की राष्ट्रीय एकात्मता के विरुद्ध काम कर रहा है।” -मोहनजी भागवत \n#RSSVijayaDashami https://t.co/Bn5cKzjeYQ",hindi_script,hin_Deva,0.999393,NaN,hin_Deva
4,"युवक हाथ जोड़ता रहा, पाँव में गिर गया, पर 500 रु की रिश्वत ना मिलने पर यूपी पुलिस की नाराज़गी का ये आलम था कि वो नहीं माने।\n\nएक पंक्ति आपने अक्सर अख़बारों में पढ़ी होगी ‘उत्तरप्रदेश में भ्रष्टाचार के प्रति ज़ीरो टालरेंस पालिसी है’। बस ये उसी का नमूना है।\n https://t.co/6PZ3WlA4tt",2127158,"युवक हाथ जोड़ता रहा, पाँव में गिर गया, पर 500 रु की रिश्वत ना मिलने पर यूपी पुलिस की नाराज़गी का ये आलम था कि वो नहीं माने।\n\nएक पंक्ति आपने अक्सर अख़बारों में पढ़ी होगी ‘उत्तरप्रदेश में भ्रष्टाचार के प्रति ज़ीरो टालरेंस पालिसी है’। बस ये उसी का नमूना है।\n https://t.co/6PZ3WlA4tt",hindi_script,hin_Deva,0.868532,NaN,hin_Deva
5,गुजरात : कच्छ के मुंद्रा इलाके में UAE से आए एक कंटेनर में 350 करोड़ की क़ीमत का 70Kg हेरोइन ड्रग्स बरामद\n\n◆पंजाब भेजा जाना था ड्रग्स का कंसाइनमेंट \n\n#Gujarat @bhupendrajourno,4917762,गुजरात : कच्छ के मुंद्रा इलाके में UAE से आए एक कंटेनर में 350 करोड़ की क़ीमत का 70Kg हेरोइन ड्रग्स बरामद\n\n◆पंजाब भेजा जाना था ड्रग्स का कंसाइनमेंट \n\n#Gujarat @bhupendrajourno,hindi_script,hin_Deva,0.999982,NaN,hin_Deva
6,मुख्यमंत्री @ChouhanShivraj ने सीधी जिले के रामपुर नैकिन के भीतरी गांव में हुई फूड प्वाइजनिंग मामले को संज्ञान में लेकर कलेक्टर को प्रभावित लोगों के समुचित इलाज और कुशलता के निर्देश दिए हैं।,6779839,मुख्यमंत्री @ChouhanShivraj ने सीधी जिले के रामपुर नैकिन के भीतरी गांव में हुई फूड प्वाइजनिंग मामले को संज्ञान में लेकर कलेक्टर को प्रभावित लोगों के समुचित इलाज और कुशलता के निर्देश दिए हैं।,hindi_script,hin_Deva,1.000040,NaN,hin_Deva
7,"गुजरात ब्रिज हादसे पर मुख्यमंत्री भूपेंद्र पटेल से सवाल पूछने पर रिपब्लिक भारत की महिला पत्रकार से बदसलूकी \n\nपत्रकार का फोन छीना गया, कैमरामैन के साथ मारपीट \n\nसवाल उठे ! ऐसा क्या छुपाया जा रहा है ? https://t.co/Fd1I0adVgb",4435984,"गुजरात ब्रिज हादसे पर मुख्यमंत्री भूपेंद्र पटेल से सवाल पूछने पर

### Roman/Hinglish branch sample

,tweet,row_id,tweet_original,translation_route,lid_lang,lid_score,xlit_lang,nllb_src_lang
0,2015 :- Delhi मैन गोव्ट हुमारी\n\n2022 :- Delhi मैन MCD भी हुमारी\n\n2024 :- Delhi मैन LG भी हुमारा ?,4590375,2015 :- Delhi main Govt humari\n\n2022 :- Delhi main MCD bhi Humari\n\n2024 :- Delhi main LG bhi humara ?,hinglish_roman,pan_Latn,0.722826,hi,hin_Deva
1,खुली नज़र क्या खेल देखेंगा दुनिया का band आँख से देख तमाशा दुनिया का I have 9 names with me. देखते कल दाल भी डेंगे. Time for #ड्रग्सफ्रीमुंबई https://t.co/U7xwu2POAm,1333700,Khuli nazar kya khel dekhenga duniya ka band aankh se dekh tamasha duniya ka I have 9 names with me. Dekhte kal daal bhi denge. Time for #DrugsFreeMumbai https://t.co/U7xwu2POAm,hinglish_roman,asm_Latn,0.755355,hi,hin_Deva
2,@PoddarVaishali @rkalia80 @TajinderBagga @HarishKhuranna @Abhiemanyu Thanks वैशाली . God ब्लेस.,2031818,@PoddarVaishali @rkalia80 @TajinderBagga @HarishKhuranna @Abhiemanyu Thanks Vaishali . God bless.,hinglish_roman,mal_Latn,5.042846,hi,hin_Deva
3,बाप बाप होता है... लेकिन बुढ़ा तोह ho hi जाता है 😭😭😭 #इंडवपाक,3476956,baap baap hota hai... lekin budhha toh ho hi jaata hai 😭😭😭 #INDvPAK,hinglish_roman,urd_Latn,6.395550,hi,hin_Deva
4,जब ज़ुल्म-o-सितम केई कोह-e-गरन रूई की तरह उड़ जाएंगे हम महकूमों केई पाओं tale ये धरती धार धार धड़केगी और अहल-e-हकम केई सार ऊपर जब बिजली कड़ कड़ कड़केगी,5727498,Jab zulm-o-sitam ke koh-e-garan\nRooi ki tarah ud jaenge\nHum mehkoomon ke paaon tale\nYe dharti dhar dhar dhadkegi\nAur ahl-e-hakam ke sar oopar\nJab bijli kad kad kadkegi,hinglish_roman,urd_Latn,6.201514,hi,hin_Deva
5,#सोरियासिफ,1636292,#SorryAsif,hinglish_roman,snd_Latn,0.791438,hi,hin_Deva
6,#न्यूप्रोफाइलपिक https://t.co/lqaI0hamIA,2826665,#NewProfilePic https://t.co/lqaI0hamIA,hinglish_roman,mal_Latn,0.698720,hi,hin_Deva
7,अवाज़ do हम ईक हैं … https://t.co/nT3swIvZnp,5087218,Awaaz do hum eik hain … https://t.co/nT3swIvZnp,hinglish_roman,urd_Latn,0.678719,hi,hin_Deva


## Translated Output Samples

These samples compare original text, normalized routed text, and final English translation.

In [5]:
translated_cols = [
    "row_id",
    "lid_lang",
    "tweet_original",
    "tweet",
    "en_text",
    "cosine_before_after",
    "translation_engine",
]

display(Markdown("### Native translated sample"))
display(sample_csv(FILES["native_translated"], n=8, usecols=translated_cols))

display(Markdown("### Roman/Hinglish translated sample"))
display(sample_csv(FILES["roman_translated"], n=8, usecols=translated_cols))

### Native translated sample

,tweet,row_id,tweet_original,lid_lang,en_text,translation_engine,cosine_before_after
0,महाराष्ट्र- कर्नाटक सीमा प्रश्न पेटवणं ही भाजपची राजकीय गरज आहे.कर्नाटकात निवडणुका होणार आहेत. कन्नड रक्षण वेदिके या संघटनेला बोम्मई सरकारचा छुपा पाठींबा दिसतो आहे. आणि आमचे मुख्यमंत्री शिंदे लाचारीत मग्न आहेत! \n#BJP #eknathshindeCM,5120331,महाराष्ट्र- कर्नाटक सीमा प्रश्न पेटवणं ही भाजपची राजकीय गरज आहे.कर्नाटकात निवडणुका होणार आहेत. कन्नड रक्षण वेदिके या संघटनेला बोम्मई सरकारचा छुपा पाठींबा दिसतो आहे. आणि आमचे मुख्यमंत्री शिंदे लाचारीत मग्न आहेत! \n#BJP #eknathshindeCM,mar_Deva,The Maharashtra-Karnataka border question is the BJP 's political need. Elections are going to be held in Karnataka. Kannada Rakhsan Vedike or the organisation seems to be the hidden backbone of the BOMME government.,nllb,0.618121
1,कानून बहुत घटिया है इसलिए 1 लाख नकली रेमडेसिविर बनाकर पूरे देश में बेच दिया\n\nनक्कालों मिलावटखोरों जमाखोरों मुनाफाखोरों और कालाबाजारियों की 100% चल-अचल संपत्ति जब्त करने तथा उन्हें आजीवन कारावास देने के लिए कानून बनाना नितांत आवश्यक है\n\n@AmitShah @blsanthosh\n@narendramodi @JPNadda,2444116,कानून बहुत घटिया है इसलिए 1 लाख नकली रेमडेसिविर बनाकर पूरे देश में बेच दिया\n\nनक्कालों मिलावटखोरों जमाखोरों मुनाफाखोरों और कालाबाजारियों की 100% चल-अचल संपत्ति जब्त करने तथा उन्हें आजीवन कारावास देने के लिए कानून बनाना नितांत आवश्यक है\n\n@AmitShah @blsanthosh\n@narendramodi @JPNadda,hin_Deva,"The law is so low that it is imperative to make laws to seize 100% of the real estate of counterfeit remdesivir smugglers, depositors, profiteers and black marketers and to sentence them to life imprisonment. @AmitShah @blsanthosh @narendramodi @JPNadda",nllb,0.751034
2,#FuelPriceHike के बारे में मोदी के मंत्री @HardeepSPuri “चलो छोड़ो यार” कह के दरवाज़ा बन्ध किया। आम आदमी कैसे छोड़ो इस बात को मंत्रीजी? कुछ तो तमीज़ सिखाओ उन्हें @PMOIndia \nhttps://t.co/jMhQknwzYF,5734683,#FuelPriceHike के बारे में मोदी के मंत्री @HardeepSPuri “चलो छोड़ो यार” कह के दरवाज़ा बन्ध किया। आम आदमी कैसे छोड़ो इस बात को मंत्रीजी? कुछ तो तमीज़ सिखाओ उन्हें @PMOIndia \nhttps://t.co/jMhQknwzYF,kas_Deva,"Modi's Minister @HardeepSPuri about #FuelPriceHike Let's go man shut the door. How can the common man let this thing go, Minister?",nllb,0.840076
3,"“राजनीतिक स्वार्थ, कट्टरपन व अलगाव की भावना, भारत के प्रति शत्रुता तथा जागतिक वर्चस्व की महत्वाकांक्षा, इनका एक अजीब सम्मिश्रण भारत की राष्ट्रीय एकात्मता के विरुद्ध काम कर रहा है।” -मोहनजी भागवत \n#RSSVijayaDashami https://t.co/Bn5cKzjeYQ",229866,"“राजनीतिक स्वार्थ, कट्टरपन व अलगाव की भावना, भारत के प्रति शत्रुता तथा जागतिक वर्चस्व की महत्वाकांक्षा, इनका एक अजीब सम्मिश्रण भारत की राष्ट्रीय एकात्मता के विरुद्ध काम कर रहा है।” -मोहनजी भागवत \n#RSSVijayaDashami https://t.co/Bn5cKzjeYQ",hin_Deva,"A strange combination of political selfishness, extreme and isolationist sentiments, hostility towards India and ambition for global dominance is working against India's national unity. - Mohenji Bhagwat #RSSVijayaDashami https://t.co/Bn5cKzjeYQ",nllb,0.921866
4,"युवक हाथ जोड़ता रहा, पाँव में गिर गया, पर 500 रु की रिश्वत ना मिलने पर यूपी पुलिस की नाराज़गी का ये आलम था कि वो नहीं माने।\n\nएक पंक्ति आपने अक्सर अख़बारों में पढ़ी होगी ‘उत्तरप्रदेश में भ्रष्टाचार के प्रति ज़ीरो टालरेंस पालिसी है’। बस ये उसी का नमूना है।\n https://t.co/6PZ3WlA4tt",2127158,"युवक हाथ जोड़ता रहा, पाँव में गिर गया, पर 500 रु की रिश्वत ना मिलने पर यूपी पुलिस की नाराज़गी का ये आलम था कि वो नहीं माने।\n\nएक पंक्ति आपने अक्सर अख़बारों में पढ़ी होगी ‘उत्तरप्रदेश में भ्रष्टाचार के प्रति ज़ीरो टालरेंस पालिसी है’। बस ये उसी का नमूना है।\n https://t.co/6PZ3WlA4tt",hin_Deva,"The young man kept holding hands, fell to his feet, but the anger of the UP police when they did not receive the Rs 500 bribe was that he did not believe. One line you have often read in the newspapers is ""There is a zero tolerance policy against corruption in Uttar Pradesh. This is just a sample of it. https://t.co/6PZ3WlA4tt",nllb,0.785015
5,गुजरात : कच्छ के मुंद्रा इलाके में UAE से आए एक कंटेनर में 350 करोड़ की क़ी

### Roman/Hinglish translated sample

,tweet,row_id,tweet_original,lid_lang,en_text,translation_engine,cosine_before_after
0,2015 :- Delhi मैन गोव्ट हुमारी\n\n2022 :- Delhi मैन MCD भी हुमारी\n\n2024 :- Delhi मैन LG भी हुमारा ?,4590375,2015 :- Delhi main Govt humari\n\n2022 :- Delhi main MCD bhi Humari\n\n2024 :- Delhi main LG bhi humara ?,pan_Latn,Delhi Man Govt Humari 2022 Delhi Man MCD also Humari 2024,nllb,0.837741
1,खुली नज़र क्या खेल देखेंगा दुनिया का band आँख से देख तमाशा दुनिया का I have 9 names with me. देखते कल दाल भी डेंगे. Time for #ड्रग्सफ्रीमुंबई https://t.co/U7xwu2POAm,1333700,Khuli nazar kya khel dekhenga duniya ka band aankh se dekh tamasha duniya ka I have 9 names with me. Dekhte kal daal bhi denge. Time for #DrugsFreeMumbai https://t.co/U7xwu2POAm,asm_Latn,I have 9 names with me. Time for #DrugFreeMumbai https://t.co/U7xwu2POAm,nllb,0.617044
2,@PoddarVaishali @rkalia80 @TajinderBagga @HarishKhuranna @Abhiemanyu Thanks वैशाली . God ब्लेस.,2031818,@PoddarVaishali @rkalia80 @TajinderBagga @HarishKhuranna @Abhiemanyu Thanks Vaishali . God bless.,mal_Latn,@PoddarVaishali @rkalia80 @TajinderBagga @HarishKhuranna @Abhiemanyu Thanks Vaishali.,nllb,0.861723
3,बाप बाप होता है... लेकिन बुढ़ा तोह ho hi जाता है 😭😭😭 #इंडवपाक,3476956,baap baap hota hai... lekin budhha toh ho hi jaata hai 😭😭😭 #INDvPAK,urd_Latn,Father is father... but old man goes to hell 😭😭😭 #induvak,nllb,0.395398
4,जब ज़ुल्म-o-सितम केई कोह-e-गरन रूई की तरह उड़ जाएंगे हम महकूमों केई पाओं tale ये धरती धार धार धड़केगी और अहल-e-हकम केई सार ऊपर जब बिजली कड़ कड़ कड़केगी,5727498,Jab zulm-o-sitam ke koh-e-garan\nRooi ki tarah ud jaenge\nHum mehkoomon ke paaon tale\nYe dharti dhar dhar dhadkegi\nAur ahl-e-hakam ke sar oopar\nJab bijli kad kad kadkegi,urd_Latn,"When the last of the oppressors will fly like a roaring wind, we will walk in the footsteps of the rulers, and the earth will be shaken, and the people will be shaken, and the lightning will be bright.",nllb,0.633683
5,#सोरियासिफ,1636292,#SorryAsif,snd_Latn,#SoriaSif,nllb,0.772401
6,#न्यूप्रोफाइलपिक https://t.co/lqaI0hamIA,2826665,#NewProfilePic https://t.co/lqaI0hamIA,mal_Latn,#NewProfilpic https://t.co/lqaI0hamIA,nllb,0.894712
7,अवाज़ do हम ईक हैं … https://t.co/nT3swIvZnp,5087218,Awaaz do hum eik hain … https://t.co/nT3swIvZnp,urd_Latn,The sound do we are eck ... https://t.co/nT3swIvZnp,nllb,0.680370


## Low-Cosine Inspection

These files are useful for manual quality checks. Lower cosine scores are more likely to contain weak or distorted translations.

In [6]:
def show_lowcos(path: Path, n: int = 10) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path)
    if "cosine_before_after" in df.columns:
        return df.sort_values("cosine_before_after", ascending=True).head(n)
    return df.head(n)


display(Markdown("### Native low-cosine examples"))
display(show_lowcos(FILES["native_lowcos"], n=10))

display(Markdown("### Roman/Hinglish low-cosine examples"))
display(show_lowcos(FILES["roman_lowcos"], n=10))

### Native low-cosine examples

,timestamp,tweet,retweet_author,original_author,retweet_lc,original_lc,retweet_party,row_id,tweet_original,translation_route,lid_lang,lid_score,lid_model,xlit_lang,nllb_src_lang,devanagari_ratio,script,en_text,translation_engine,cosine_before_after
293487,2022-12-23 11:17:40+00:00,"సినీ హీరోలతో పోటీపడుతూ చాలా అద్భుతంగా నటించే వారాయన. ఏ పాత్ర ఇచ్చినా ఆ పాత్రలో నటించి, అద్భుతమైన పేరు తెచ్చుకున్న వ్యక్తి. నేను కొంతకాలం వారితో కలిసి పనిచేయడం జరిగింది. ఆ కాలంలో వారితో కొన్ని అనుభవాలను కూడా పంచుకున్నాం. తెలుగు చలనచిత్ర పరిశ్రమ కైకాల గారిని కోల్పోవడం బాధాకరం: సీఎం",YadavTalasani,TelanganaCMO,yadavtalasani,telanganacmo,TRS,5151545,"సినీ హీరోలతో పోటీపడుతూ చాలా అద్భుతంగా నటించే వారాయన. ఏ పాత్ర ఇచ్చినా ఆ పాత్రలో నటించి, అద్భుతమైన పేరు తెచ్చుకున్న వ్యక్తి. నేను కొంతకాలం వారితో కలిసి పనిచేయడం జరిగింది. ఆ కాలంలో వారితో కొన్ని అనుభవాలను కూడా పంచుకున్నాం. తెలుగు చలనచిత్ర పరిశ్రమ కైకాల గారిని కోల్పోవడం బాధాకరం: సీఎం",hindi_script,tel_Telu,1.000049,IndicLID-FTN,NaN,tel_Telu,0.000000,roman_hindi,He was a very talented actor who competed with film heroes. He was a great actor who made a name for himself in any role. I worked with him for some time. We also shared some experiences with him during that time.,nllb,-0.150901
299009,2022-12-23 12:45:27+00:00,"సినీ హీరోలతో పోటీపడుతూ చాలా అద్భుతంగా నటించే వారాయన. ఏ పాత్ర ఇచ్చినా ఆ పాత్రలో నటించి, అద్భుతమైన పేరు తెచ్చుకున్న వ్యక్తి. నేను కొంతకాలం వారితో కలిసి పనిచేయడం జరిగింది. ఆ కాలంలో వారితో కొన్ని అనుభవాలను కూడా పంచుకున్నాం. తెలుగు చలనచిత్ర పరిశ్రమ కైకాల గారిని కోల్పోవడం బాధాకరం: సీఎం",NKB_BRS,TelanganaCMO,nkb_brs,telanganacmo,NaN,5220147,"సినీ హీరోలతో పోటీపడుతూ చాలా అద్భుతంగా నటించే వారాయన. ఏ పాత్ర ఇచ్చినా ఆ పాత్రలో నటించి, అద్భుతమైన పేరు తెచ్చుకున్న వ్యక్తి. నేను కొంతకాలం వారితో కలిసి పనిచేయడం జరిగింది. ఆ కాలంలో వారితో కొన్ని అనుభవాలను కూడా పంచుకున్నాం. తెలుగు చలనచిత్ర పరిశ్రమ కైకాల గారిని కోల్పోవడం బాధాకరం: సీఎం",hindi_script,tel_Telu,1.000049,IndicLID-FTN,NaN,tel_Telu,0.000000,roman_hindi,He was a very talented actor who competed with film heroes. He was a great actor who made a name for himself in any role. I worked with him for some time. We also shared some experiences with him during that time.,nllb,-0.150901
248376,2022-04-15 04:35:23+00:00,"గతంలో మాదిరిగానే కనీస మద్దతు ధర చెల్లించి, రాష్ట్ర ప్రభుత్వమే ధాన్యం కొనుగోలు చేస్తుంది కనుక రాష్ట్రంలోని రైతులెవరూ తక్కువ ధరకు ధాన్యం ఇతరులకు అమ్మి, నష్టపోవద్దని ముఖ్యమంత్రి సూచించారు.",KaleemTRS,TelanganaCMO,kaleemtrs,telanganacmo,TRS,4358326,"గతంలో మాదిరిగానే కనీస మద్దతు ధర చెల్లించి, రాష్ట్ర ప్రభుత్వమే ధాన్యం కొనుగోలు చేస్తుంది కనుక రాష్ట్రంలోని రైతులెవరూ తక్కువ ధరకు ధాన్యం ఇతరులకు అమ్మి, నష్టపోవద్దని ముఖ్యమంత్రి సూచించారు.",hindi_script,tel_Telu,1.000049,IndicLID-FTN,NaN,tel_Telu,0.000000,roman_hindi,"The Chief Minister suggested that no farmer in the state should sell the grain at a lower price to others, so that the state government would buy the grain at the same minimum support price as before.",nllb,-0.149682
298997,2022-12-23 16:38:15+00:00,"ఇదే సూత్రాన్ని అనుసరించి సమస్త రంగాల్లో వృద్ధిరేటు వూహించని రీతిలో నమోదవుతూ, అటు తలసరి ఆదాయం, ఇటు జీఎస్డీపీ పెరుగుదలకు దోహదం చేసిందని సీఎం అన్నారు.",NKB_BRS,TelanganaCMO,nkb_brs,telanganacmo,NaN,5220132,"ఇదే సూత్రాన్ని అనుసరించి సమస్త రంగాల్లో వృద్ధిరేటు వూహించని రీతిలో నమోదవుతూ, అటు తలసరి ఆదాయం, ఇటు జీఎస్డీపీ పెరుగుదలకు దోహదం చేసిందని సీఎం అన్నారు.",hindi_script,tel_Telu,1.000049,IndicLID-FTN,NaN,tel_Telu,0.000000,roman_hindi,"Following the same principle, the growth rate in all sectors has been unpredictable, and this has contributed to the growth of Gross Domestic Product (GDP) and Gross Domestic Product (GDP).",nllb,-0.131535
362045,2023-04-11 10:41:12+00:00,”எம்எல்ஏக்களுக்கு ஐபிஎல் பாஸ் - எஸ்பி வேலுமணி கோரிக்கை”\n\nhttps://t.co/WciCN2SQmv | #SPVelumani | #IPL | #News7Tamil | #News7TamilUpdates | #UdhayanidhiStalin | #TNAssembly https://t.co/y2pGrk4BMY,Dhilibandmk,news7tamil,dhilibandmk,news7tamil,DMK,6267270,”எம்எல்ஏக்களுக்கு ஐபிஎல் பாஸ் - எஸ்பி வேலுமணி கோரிக்கை”\n\nhttps://t.co/WciCN2SQmv | #SPVeluman

### Roman/Hinglish low-cosine examples

,timestamp,tweet,retweet_author,original_author,retweet_lc,original_lc,retweet_party,row_id,tweet_original,translation_route,lid_lang,lid_score,lid_model,xlit_lang,nllb_src_lang,devanagari_ratio,script,en_text,translation_engine,cosine_before_after
2012,2021-12-09 09:32:25+00:00,जीओए ❤️ @ArvindKejriwal https://t.co/4Qltj6UUC4,priyankaaap23,Amitjanhit,priyankaaap23,amitjanhit,AAP,3815528,Goa ❤️ @ArvindKejriwal https://t.co/4Qltj6UUC4,hinglish_roman,kok_Latn,0.690180,IndicLID-FTR,hi,hin_Deva,0.085106,roman_hindi,"The first is the ""Goldilocks"".",nllb,-0.115032
2975,2023-05-08 04:20:19+00:00,@UmarPat87072613 @shakilpatel86 @OICatUN @LadyVelvet_HFQ @HamdanMohammed @AllahGreatQuran @KingSalman @AliyaAbbas @alkawari4unesco @s_szama @aajtak @ABPNews @DibluKaka @DHOLUKHA8 @BBCHindi @AtmaramSoni3 @AnsariK786 @AnoopMaheshwar5 @AdarshK22237787 @Aafiya_khan_786 @Aam_Nagrik2315 @_MS_Ali_Rana @aapsur @alarafatengr @ALLAUDIENKHAN1 @Stanalb @ZulekhaJehan @MdtajKohri5 @Mansoor19302780 @Zaki_Sohail_INC @ExHindu07006862 @kf07b @Dhanpal85554293 @DevkarnMeenaINC @Deol123456788 @DeccanEnglish @DAS89764172 @CRUZONTIVEROS11 @creterion @BisnoiRamkaran @tanab786 वालेकुम अस्सलम,s_szama,alamgirizvi,s_szama,alamgirizvi,INC,6531389,@UmarPat87072613 @shakilpatel86 @OICatUN @LadyVelvet_HFQ @HamdanMohammed @AllahGreatQuran @KingSalman @AliyaAbbas @alkawari4unesco @s_szama @aajtak @ABPNews @DibluKaka @DHOLUKHA8 @BBCHindi @AtmaramSoni3 @AnsariK786 @AnoopMaheshwar5 @AdarshK22237787 @Aafiya_khan_786 @Aam_Nagrik2315 @_MS_Ali_Rana @aapsur @alarafatengr @ALLAUDIENKHAN1 @Stanalb @ZulekhaJehan @MdtajKohri5 @Mansoor19302780 @Zaki_Sohail_INC @ExHindu07006862 @kf07b @Dhanpal85554293 @DevkarnMeenaINC @Deol123456788 @DeccanEnglish @DAS89764172 @CRUZONTIVEROS11 @creterion @BisnoiRamkaran @tanab786 Walekum Assalam,hinglish_roman,san_Latn,3.560821,IndicLID-BERT,hi,hin_Deva,0.022688,roman_hindi,The first is the fact that the government has not yet made a decision on the issue of the issue of the right to vote. The first is that the government has not yet made a decision on the issue of the right to vote.,nllb,-0.104374
2478,2022-02-07 09:09:43+00:00,@ParveenKader @sursmi @MehekF @kauserseema @p_iqbalkhan @s_szama @mazhar_jafri @INCAseemKP @asimtwitt @1Patelzuber @Lasani309 @tanveer1729 @INCObaid @Raheodisha @AwasimkKhan वालेकुम अस्सलम.,s_szama,alamgirizvi,s_szama,alamgirizvi,INC,4947961,@ParveenKader @sursmi @MehekF @kauserseema @p_iqbalkhan @s_szama @mazhar_jafri @INCAseemKP @asimtwitt @1Patelzuber @Lasani309 @tanveer1729 @INCObaid @Raheodisha @AwasimkKhan Waleykum Assalam.,hinglish_roman,kas_Latn,5.717268,IndicLID-BERT,hi,hin_Deva,0.068783,roman_hindi,The first is the fact that the country is not a part of the Union.,nllb,-0.100057
2598,2022-03-09 14:44:29+00:00,@mkstalin @CMOTamilnadu @arivalayam @Udhaystalin @PrabhuChawla @manojksonthalia @ptrmadurai @KPonmudiMLA @TRBRajaa @KanimozhiDMK @evvelu @TThenarasu @KN_NEHRU @V_Senthilbalaji,Amirkhansurya,NewIndianXpress,amirkhansurya,newindianxpress,DMK,5254466,@mkstalin @CMOTamilnadu @arivalayam @Udhaystalin @PrabhuChawla @manojksonthalia @ptrmadurai @KPonmudiMLA @TRBRajaa @KanimozhiDMK @evvelu @TThenarasu @KN_NEHRU @V_Senthilbalaji,hinglish_roman,tam_Latn,5.293069,IndicLID-BERT,hi,hin_Deva,0.000000,roman_hindi,The first is the fact that the government has not yet taken any action to stop the spread of the virus.,nllb,-0.094628
2846,2022-03-09 12:42:13+00:00,@mkstalin @CMOTamilnadu @arivalayam @Udhaystalin @PrabhuChawla @manojksonthalia @ptrmadurai @KPonmudiMLA @TRBRajaa @KanimozhiDMK @evvelu @TThenarasu @KN_NEHRU @V_Senthilbalaji,PANIYAZAHMED2,NewIndianXpress,paniyazahmed2,newindianxpress,DMK,6088435,@mkstalin @CMOTamilnadu @arivalayam @Udhaystalin @PrabhuChawla @manojksonthalia @ptrmadurai @KPonmudiMLA @TRBRajaa @KanimozhiDMK @evvelu @TThenarasu @KN_NEHRU @V_Senthilbalaji,hinglish_roman,tam_Latn,5.293069,IndicLID-BERT,hi,hin_Deva,0.000000,roman_hindi,The first is the fact that the government has not yet taken any action to sto

## Row Lookup

Use `lookup_row(row_id)` to inspect one row across every available stage.

In [7]:
row_id_to_check = 166
row_hits = lookup_row(row_id_to_check)
for name, df in row_hits.items():
    display(Markdown(f"### {name}"))
    display(df)

/tmp/ipykernel_1800967/2844409666.py:37: DtypeWarning: Columns (2,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(path, chunksize=100_000):
/tmp/ipykernel_1800967/2844409666.py:37: DtypeWarning: Columns (2,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(path, chunksize=100_000):
/tmp/ipykernel_1800967/2844409666.py:37: DtypeWarning: Columns (2,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(path, chunksize=100_000):
/tmp/ipykernel_1800967/2844409666.py:37: DtypeWarning: Columns (2,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(path, chunksize=100_000):
/tmp/ipykernel_1800967/2844409666.py:37: DtypeWarning: Columns (2,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(path, chunksize=

### roman_routed

,timestamp,tweet,retweet_author,original_author,retweet_lc,original_lc,retweet_party,row_id,tweet_original,translation_route,lid_lang,lid_score,lid_model,xlit_lang,nllb_src_lang
0,2020-10-16 10:05:20+00:00,Oscar पुरस्कार से सम्मानित मशहूर कॉस्ट्यूम designer भानू अथैया जी का आज स्वर्गवास हुआ. सुनके मुझे बहुत दुख हुआ. गांधी film केई लिये उन्हे Oscar से सम्मानित किया गया था. उन्होंने हमारी Picture लेकिन केई लिये भी कॉस्ट्यूम डिजाइनिंग की थी. ईश्वर उनकी आत्मा को शांति de. https://t.co/JMB9a4o1n5,nlshmishra5,mangeshkarlata,nlshmishra5,mangeshkarlata,BJP,166,Oscar puraskar se sammanit mashhoor costume designer Bhanu Athaiya ji ka aaj swargwas hua. Sunke mujhe bahut dukh hua. Gandhi film ke liye unhe Oscar se sammanit kiya gaya tha. Unhone hamari Picture Lekin ke liye bhi costume Designing ki thi. Ishwar unki aatma ko shanti de. https://t.co/JMB9a4o1n5,hinglish_roman,hin_Latn,9.102791,IndicLID-BERT,hi,hin_Deva


### roman_translated

,timestamp,tweet,retweet_author,original_author,retweet_lc,original_lc,retweet_party,row_id,tweet_original,translation_route,lid_lang,lid_score,lid_model,xlit_lang,nllb_src_lang,devanagari_ratio,script,en_text,translation_engine,cosine_before_after
0,2020-10-16 10:05:20+00:00,Oscar पुरस्कार से सम्मानित मशहूर कॉस्ट्यूम designer भानू अथैया जी का आज स्वर्गवास हुआ. सुनके मुझे बहुत दुख हुआ. गांधी film केई लिये उन्हे Oscar से सम्मानित किया गया था. उन्होंने हमारी Picture लेकिन केई लिये भी कॉस्ट्यूम डिजाइनिंग की थी. ईश्वर उनकी आत्मा को शांति de. https://t.co/JMB9a4o1n5,nlshmishra5,mangeshkarlata,nlshmishra5,mangeshkarlata,BJP,166,Oscar puraskar se sammanit mashhoor costume designer Bhanu Athaiya ji ka aaj swargwas hua. Sunke mujhe bahut dukh hua. Gandhi film ke liye unhe Oscar se sammanit kiya gaya tha. Unhone hamari Picture Lekin ke liye bhi costume Designing ki thi. Ishwar unki aatma ko shanti de. https://t.co/JMB9a4o1n5,hinglish_roman,hin_Latn,9.102791,IndicLID-BERT,hi,hin_Deva,0.634483,devanagari,I was very sad to hear that the Oscar winning costume designer Bhanu Atayyaji was in heaven today. He was awarded the Oscar for Gandhi film. He designed costumes for our Picture but also for someone.,nllb,0.813009


### merged

,timestamp,tweet,retweet_author,original_author,retweet_lc,original_lc,retweet_party,row_id,tweet_original,translation_route,lid_lang,lid_score,lid_model,xlit_lang,nllb_src_lang,devanagari_ratio,script,en_text,translation_engine,cosine_before_after
0,2020-10-16 10:05:20+00:00,Oscar पुरस्कार से सम्मानित मशहूर कॉस्ट्यूम designer भानू अथैया जी का आज स्वर्गवास हुआ. सुनके मुझे बहुत दुख हुआ. गांधी film केई लिये उन्हे Oscar से सम्मानित किया गया था. उन्होंने हमारी Picture लेकिन केई लिये भी कॉस्ट्यूम डिजाइनिंग की थी. ईश्वर उनकी आत्मा को शांति de. https://t.co/JMB9a4o1n5,nlshmishra5,mangeshkarlata,nlshmishra5,mangeshkarlata,BJP,166,Oscar puraskar se sammanit mashhoor costume designer Bhanu Athaiya ji ka aaj swargwas hua. Sunke mujhe bahut dukh hua. Gandhi film ke liye unhe Oscar se sammanit kiya gaya tha. Unhone hamari Picture Lekin ke liye bhi costume Designing ki thi. Ishwar unki aatma ko shanti de. https://t.co/JMB9a4o1n5,hinglish_roman,hin_Latn,9.102791,IndicLID-BERT,hi,hin_Deva,0.634483,devanagari,I was very sad to hear that the Oscar winning costume designer Bhanu Atayyaji was in heaven today. He was awarded the Oscar for Gandhi film. He designed costumes for our Picture but also for someone.,nllb,0.813009


## Text Search

Use this to find tweets containing a specific term or phrase in the original text.

In [8]:
search_pattern = "Oscar|Farmer|मोदी"
display(search_text(FILES["native_routed"], search_pattern, text_col="tweet_original", n=10))

,timestamp,tweet,retweet_author,original_author,retweet_lc,original_lc,retweet_party,row_id,tweet_original,translation_route,lid_lang,lid_score,lid_model,xlit_lang,nllb_src_lang
0,2020-12-15 09:47:14+00:00,VIDEO : Pune | केंद्रीय मंत्री प्रकाश जावडेकरांच्या घरासमोर शेतकऱ्यांचं आंदोलन\n#Pune #PrakashJavdekar #Farmer @PrakashJavdekar https://t.co/yIAmN8b2Ah,VartakKuldeep,TV9Marathi,vartakkuldeep,tv9marathi,INC,4,VIDEO : Pune | केंद्रीय मंत्री प्रकाश जावडेकरांच्या घरासमोर शेतकऱ्यांचं आंदोलन\n#Pune #PrakashJavdekar #Farmer @PrakashJavdekar https://t.co/yIAmN8b2Ah,hindi_script,mar_Deva,0.992384,IndicLID-FTN,NaN,mar_Deva
1,2020-12-29 11:39:12+00:00,पश्चिम बंगालच्या बीरभूममध्ये मंगळवारी राज्याच्या मुख्यमंत्री आणि तृणमूल काँग्रेसच्या प्रमुख ममता बनर्जी यांनी पदयात्रा काढली. यानंतर झालेल्या सभेत त्यांनी भाजप आणि पंतप्रधान मोदी यांच्यावर जोरदार हल्ला चढवला. https://t.co/BgkY81yDCo,VartakKuldeep,SaamanaOnline,vartakkuldeep,saamanaonline,INC,6,पश्चिम बंगालच्या बीरभूममध्ये मंगळवारी राज्याच्या मुख्यमंत्री आणि तृणमूल काँग्रेसच्या प्रमुख ममता बनर्जी यांनी पदयात्रा काढली. यानंतर झालेल्या सभेत त्यांनी भाजप आणि पंतप्रधान मोदी यांच्यावर जोरदार हल्ला चढवला. https://t.co/BgkY81yDCo,hindi_script,mar_Deva,1.000049,IndicLID-FTN,NaN,mar_Deva
2,2020-08-08 17:54:01+00:00,मोदी सरकारने घोषित केलेले 20 लाख कोटी कुठे गेले असा प्रश्न उपस्थित करत महाराष्ट्र युवक काँग्रेस एक अनोखं राज्यव्यापी आंदोलन करणार आहे.\n#pmmodi @satyajeettambe @BJP4Maharashtra @INCMaharashtra \n\nhttps://t.co/B9sX5PwoO7,VartakKuldeep,abpmajhatv,vartakkuldeep,abpmajhatv,INC,16,मोदी सरकारने घोषित केलेले 20 लाख कोटी कुठे गेले असा प्रश्न उपस्थित करत महाराष्ट्र युवक काँग्रेस एक अनोखं राज्यव्यापी आंदोलन करणार आहे.\n#pmmodi @satyajeettambe @BJP4Maharashtra @INCMaharashtra \n\nhttps://t.co/B9sX5PwoO7,hindi_script,mar_Deva,0.999789,IndicLID-FTN,NaN,mar_Deva
3,2020-07-08 12:35:47+00:00,"लंडन, यूएईमधील फ्लॅटसह नीरव मोदीची ३३० कोटीची संपत्ती जप्त \nhttps://t.co/6tlzXyFv9i \nईडीचा नीरव मोदीला दणका via @LoksattaLive \n#NiravModi #ED #PNBScam",VartakKuldeep,LoksattaLive,vartakkuldeep,loksattalive,INC,20,"लंडन, यूएईमधील फ्लॅटसह नीरव मोदीची ३३० कोटीची संपत्ती जप्त \nhttps://t.co/6tlzXyFv9i \nईडीचा नीरव मोदीला दणका via @LoksattaLive \n#NiravModi #ED #PNBScam",hindi_script,mar_Deva,0.995243,IndicLID-FTN,NaN,mar_Deva
4,2020-10-20 06:37:35+00:00,जीएसटी पर मोदी सत्ता का यू टर्न....\nविपक्ष के लिए सीख है आपसी एकता रखे..\nबधाई छत्तीसगढ़ के जीएसटी मंत्री @TS_SinghDeo जी...\nज़ोर-शोर से मुद्दा उठाया सफलता मिली... https://t.co/65gZdrMO0y,ShivsahayRana,ppbajpai,shivsahayrana,ppbajpai,INC,42,जीएसटी पर मोदी सत्ता का यू टर्न....\nविपक्ष के लिए सीख है आपसी एकता रखे..\nबधाई छत्तीसगढ़ के जीएसटी मंत्री @TS_SinghDeo जी...\nज़ोर-शोर से मुद्दा उठाया सफलता मिली... https://t.co/65gZdrMO0y,hindi_script,hin_Deva,0.992169,IndicLID-FTN,NaN,hin_Deva
5,2020-09-04 05:43:18+00:00,—23.9 % जीडीपी\nआज़ादी के बाद इक्नामी की सबसे बुरी गत...\nमान लिजिए मोदी देश चलाना नहीं जानते...! \nhttps://t.co/VM2uPpmmO2 via @YouTube,ShivsahayRana,ppbajpai,shivsahayrana,ppbajpai,INC,50,—23.9 % जीडीपी\nआज़ादी के बाद इक्नामी की सबसे बुरी गत...\nमान लिजिए मोदी देश चलाना नहीं जानते...! \nhttps://t.co/VM2uPpmmO2 via @YouTube,hindi_script,hin_Deva,0.880143,IndicLID-FTN,NaN,hin_Deva
6,2020-12-16 04:34:49+00:00,"त्रिपुरा के CM @BjpBiplab बोले, किसान को कम्युनिस्ट पार्टियां गुमराह कर रही हैं #हल्ला_बोल #FarmersProtest \n@anjanaomkashyap https://t.co/jz5OiGA1tr",nlshmishra5,aajtak,nlshmishra5,aajtak,BJP,100,"त्रिपुरा के CM @BjpBiplab बोले, किसान को कम्युनिस्ट पार्टियां गुमराह कर रही हैं #हल्ला_बोल #FarmersProtest \n@anjanaomkashyap https://t.co/jz5OiGA1tr",hindi_script,hin_Deva,0.439642,IndicLID-FTN,NaN,hin_Deva
7,2020-12-23 16:22:06+00:00,Video : Atul Bhatkhalkar | घराणेशाहीच्या तत्वावर चालणाऱ्यांनी मोदींना सल्ला देऊ नये - अतुल भातखळकर\n\n#Mumbai @BhatkhalkarA https://t.co/h6TfDmQKXP,nlshmishra5,TV9Marathi,nlshmishra5,tv9marathi,BJP,102,Video : Atul Bhatkhalkar | घराणेशाहीच्या तत्वावर चालणाऱ्यांनी मोदींना सल्ला देऊ नये - अतुल भातखळकर\n\n#Mumbai @BhatkhalkarA htt

## Final Merged Output

This will only display rows once the final merge file exists.

In [9]:
merged_cols = [
    "row_id",
    "translation_route",
    "tweet_original",
    "en_text",
    "translation_engine",
]

if FILES["merged"].exists():
    display(head_csv(FILES["merged"], n=10, usecols=merged_cols))
else:
    display(Markdown("Merged file does not exist yet. Rerun this cell after the merge stage finishes."))

,row_id,tweet_original,translation_route,en_text,translation_engine
0,0,PM ⁦@narendramodi⁩ to launch Garib Kalyan Rojgar Abhiyaan on 20th June to boost livelihood opportunities in Rural India. \nhttps://t.co/zCG7yxUaGq\n\nvia NaMo App https://t.co/4i34NRHCYv,english,PM ⁦@narendramodi⁩ to launch Garib Kalyan Rojgar Abhiyaan on 20th June to boost livelihood opportunities in Rural India. \nhttps://t.co/zCG7yxUaGq\n\nvia NaMo App https://t.co/4i34NRHCYv,unchanged
1,1,"A step towards economic resilience #aatmanirbharbharat, the FME scheme-with an outlay of 10,000 Crores shall seek to leverage investment worth 25000 Crore and generate gainful #employment for 9 lac people over the next 5 years.\n\nGet ready to script your food story. https://t.co/JC4Jk7xsLY",english,"A step towards economic resilience #aatmanirbharbharat, the FME scheme-with an outlay of 10,000 Crores shall seek to leverage investment worth 25000 Crore and generate gainful #employment for 9 lac people over the next 5 years.\n\nGet ready to script your food story. https://t.co/JC4Jk7xsLY",unchanged
2,2,.@MOFPI_GOI wholeheartedly welcomes the Cabinet's decision on scheme for Formalisation of Micro food processing Enterprises (#FME) which shall provide a big boost to the SHGs/FPOs/Cooperatives &amp; micro food businesses and ensure adequate #livelihoods for rural-urban communities. https://t.co/2xOPJyjEQ0,english,.@MOFPI_GOI wholeheartedly welcomes the Cabinet's decision on scheme for Formalisation of Micro food processing Enterprises (#FME) which shall provide a big boost to the SHGs/FPOs/Cooperatives &amp; micro food businesses and ensure adequate #livelihoods for rural-urban communities. https://t.co/2xOPJyjEQ0,unchanged
3,3,राज्यपाल भगतसिंह कोश्यारी यांनी मुख्यमंत्री उद्धव ठाकरे यांना पत्र लिहून विद्यापीठातील विद्यार्थ्यांच्या अंतिम वर्षाच्या परीक्षा घेण्यासंदर्भात महाराष्ट्र सार्वजनिक विद्यापीठ कायद्यातील तरतुदीनुसार निर्णय घेण्यात येईल असे कळविले आहे.,hindi_script,Governor Bhagat Singh Koshyari has written to Chief Minister Uddhav Thakere to inform him that a decision will be taken in accordance with the provisions of the Maharashtra Public Universities Act regarding the final year examination of students of the university.,nllb
4,4,VIDEO : Pune | केंद्रीय मंत्री प्रकाश जावडेकरांच्या घरासमोर शेतकऱ्यांचं आंदोलन\n#Pune #PrakashJavdekar #Farmer @PrakashJavdekar https://t.co/yIAmN8b2Ah,hindi_script,VIDEO: Pune: Farmers' movement in front of the house of Union Minister Prakash Javadekar #Pune #PrakashJavdekar #Farmer @PrakashJavdekar https://t.co/yIAmN8b2Ah,nllb
5,5,"'लडका छोटा हैं तो क्या हुआ?, बडा हमें ही करना हैं', सत्यजीत तांबेंसाठी अहमद पटेलांचा थेट ठाकरेंना फोन\nhttps://t.co/udNWWVywm8\n@satyajeettambe @INCMaharashtra #Congress",hindi_script,"'What happened when the kids are small, the big ones we have to do', Ahmed Patel's direct call to Thakarena for Satyajit Tambeng https://t.co/udNWWVywm8 @satyajeettambe @INCMaharashtra #Congress",nllb
6,6,पश्चिम बंगालच्या बीरभूममध्ये मंगळवारी राज्याच्या मुख्यमंत्री आणि तृणमूल काँग्रेसच्या प्रमुख ममता बनर्जी यांनी पदयात्रा काढली. यानंतर झालेल्या सभेत त्यांनी भाजप आणि पंतप्रधान मोदी यांच्यावर जोरदार हल्ला चढवला. https://t.co/BgkY81yDCo,hindi_script,"The Chief Minister of the state of West Bengal and the head of the Trinamool Congress, Mamata Banerjee, marched to Birbhum, West Bengal.",nllb
7,7,Russia agrees to produce coronavirus vaccine Sputnik V in India https://t.co/Xw8qbUU1Y1 https://t.co/6YTE5LQFsI,english,Russia agrees to produce coronavirus vaccine Sputnik V in India https://t.co/Xw8qbUU1Y1 https://t.co/6YTE5LQFsI,unchanged
8,8,"Overwhelmed and deeply moved by everyone’s condolence messages through various channels.I’m sorry for not being able to respond immediately especially on whatsapp. I will do so slowly over time. Please pray for my papa. May Allah grant him jannat-e-firdaus, Inshallah",english,"Overwhelmed and deeply moved by everyone’s condolence messages through various channels.I’m sorry for not being able to 